# Road Following Live (JetRacer ONNX + ROS Subscriber Pipeline)

This notebook subscribes to the **ROS Camera Topic** (`/csi_cam_0/image_raw`) to avoid hardware CSI Camera device conflicts, executing ONNX model inference and **Stanley / PID Control** on the physical JetRacer.

### 1. Setup Environment & Load ONNX Model

In [ ]:
import os
import sys
from pathlib import Path

# Add parent directory to sys.path to access Controller.py, Runner.py, and utils.py
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

import onnxruntime as ort
try:
    from jetracer.utils import preprocess_onnx, bgr8_to_jpeg
except ImportError:
    from utils import preprocess_onnx, bgr8_to_jpeg

# Locate ONNX model file
model_path = os.path.join(Path.cwd(), "road_following_model.onnx")
if not os.path.exists(model_path):
    model_path = os.path.join(parent_dir, "notebooks", "road_following_model.onnx")

if not os.path.exists(model_path):
    print(f"[!] ERROR: ONNX model file '{model_path}' not found!")
else:
    print(f"[*] Loading ONNX model from: {model_path}")

available_providers = ort.get_available_providers()
providers = ['CUDAExecutionProvider'] if 'CUDAExecutionProvider' in available_providers else []
providers.append('CPUExecutionProvider')

try:
    session = ort.InferenceSession(model_path, providers=providers)
except Exception:
    session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name
print(f"[+] Loaded ONNX Session with providers: {session.get_providers()}")


### 2. Initialize ROS Node & JetRacer Hardware (`NvidiaRacecar`)

In [ ]:
import rospy
from sensor_msgs.msg import Image
from jetracer.nvidia_racecar import NvidiaRacecar
try:
    from jetracer.Controller import StanleyController, PIDController
    from jetracer.Runner import JetRacerROSOnnxRunner
except ImportError:
    from Controller import StanleyController, PIDController
    from Runner import JetRacerROSOnnxRunner

# 1. Initialize ROS Node
try:
    rospy.init_node('road_following_live_notebook', anonymous=True, disable_signals=True)
    print("[+] ROS Node initialized successfully!")
except Exception as e:
    print(f"[*] ROS Node notice: {e}")

# 2. Hardware & Controller Setup
car = NvidiaRacecar()
stanley = StanleyController()
pid = PIDController()
stanley.reset()
pid.reset()

print("[+] JetRacer hardware and Stanley/PID Controllers initialized.")


### 3. Interactive Slider Controls & ROS Topic Live Drive Workflow (Replacing Parser Args)

In [ ]:
import cv2
import ipywidgets
import traitlets
import time
import json
from IPython.display import display
from ipywidgets import Layout

slider_style = {'description_width': '140px'}

# Control Sliders matching ROS ONNX Stanley workflow (replacing argparse args)
k_stanley_slider     = ipywidgets.FloatSlider(description='Stanley Gain (k)', min=0.1, max=3.0, value=2.5, step=0.05, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=slider_style)
base_throttle_slider = ipywidgets.FloatSlider(description='Base Throttle', min=0.05, max=0.6, value=0.20, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=slider_style)
brake_gain_slider    = ipywidgets.FloatSlider(description='Brake Gain', min=0.0, max=0.5, value=0.10, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=slider_style)
steering_bias_slider = ipywidgets.FloatSlider(description='Steering Bias', min=-0.5, max=0.5, value=0.0, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=slider_style)
alpha_slider         = ipywidgets.FloatSlider(description='Alpha (Kalman)', min=0.1, max=1.0, value=0.4, step=0.05, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=slider_style)

# Optional PID Tuning Sliders
kp_slider            = ipywidgets.FloatSlider(description='Kp (PID)', min=0.0, max=3.0, value=1.0, step=0.05, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=slider_style)
ki_slider            = ipywidgets.FloatSlider(description='Ki (PID)', min=0.0, max=0.5, value=0.0, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=slider_style)
kd_slider            = ipywidgets.FloatSlider(description='Kd (PID)', min=0.0, max=1.0, value=0.15, step=0.02, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=slider_style)

# Live Output Displays (Read-only)
raw_x_disp           = ipywidgets.FloatSlider(description='Network Target X', min=-1.0, max=1.0, value=0.0, step=0.01, readout=True, readout_format='.3f', disabled=True, layout=Layout(width='400px'), style=slider_style)
steering_disp        = ipywidgets.FloatSlider(description='Live Steering', min=-1.0, max=1.0, value=0.0, step=0.01, readout=True, readout_format='.3f', disabled=True, layout=Layout(width='400px'), style=slider_style)
throttle_disp        = ipywidgets.FloatSlider(description='Live Throttle', min=0.0, max=0.6, value=0.0, step=0.01, readout=True, readout_format='.3f', disabled=True, layout=Layout(width='400px'), style=slider_style)

# Mode Selector & Controls
state_widget         = ipywidgets.ToggleButtons(options=['Off', 'Stanley Live Drive', 'PID Live Drive'], description='Mode', value='Off')
prediction_widget   = ipywidgets.Image(format='jpeg', width=224, height=224)
reset_button         = ipywidgets.Button(description='Stop & Reset', button_style='warning', icon='refresh')
load_config_button   = ipywidgets.Button(description='Load Config JSON', button_style='info', icon='download')

def load_best_config(b=None):
    cfg_path = os.path.join(Path.cwd(), 'best_pid_config.json')
    if not os.path.exists(cfg_path):
        cfg_path = os.path.join(parent_dir, 'notebooks', 'best_pid_config.json')
    if os.path.exists(cfg_path):
        try:
            with open(cfg_path, 'r', encoding='utf-8') as f:
                cfg = json.load(f)
            k_stanley_slider.value     = cfg.get('k', k_stanley_slider.value)
            base_throttle_slider.value = cfg.get('base_throttle', base_throttle_slider.value)
            brake_gain_slider.value    = cfg.get('brake_gain', brake_gain_slider.value)
            steering_bias_slider.value = cfg.get('bias', steering_bias_slider.value)
            alpha_slider.value         = cfg.get('alpha', alpha_slider.value)
            kp_slider.value            = cfg.get('kp', kp_slider.value)
            ki_slider.value            = cfg.get('ki', ki_slider.value)
            kd_slider.value            = cfg.get('kd', kd_slider.value)
            print(f"[+] Loaded configuration from '{cfg_path}'!")
        except Exception as e:
            print(f"[!] Error loading config: {e}")
    else:
        print(f"[!] Config file '{cfg_path}' not found!")

load_config_button.on_click(load_best_config)

# Frame Callback for Live ROS Topic Stream & ONNX Inference
def on_ros_frame(cv_image, raw_x, raw_y, smoothed_x, steering, dyn_throttle):
    mode = state_widget.value
    if mode == 'Off':
        return
    
    if mode == 'PID Live Drive':
        steering = pid.update(
            raw_x=raw_x,
            kp=kp_slider.value,
            ki=ki_slider.value,
            kd=kd_slider.value,
            alpha=alpha_slider.value,
            bias=steering_bias_slider.value
        )
        dyn_throttle = base_throttle_slider.value - brake_gain_slider.value * abs(steering)
        dyn_throttle = max(0.05, min(0.5, dyn_throttle))
        car.steering = steering
        car.throttle = dyn_throttle
        smoothed_x = pid.smoothed_x

    raw_x_disp.value = raw_x
    steering_disp.value = steering
    throttle_disp.value = dyn_throttle

    # Draw target circle on frame preview
    h, w = cv_image.shape[:2]
    px = int(w * (smoothed_x / 2.0 + 0.5))
    py = int(h * (raw_y / 2.0 + 0.5)) if raw_y != 0.0 else int(h * 0.5)

    prediction = cv_image.copy()
    cv2.circle(prediction, (px, py), 8, (0, 255, 0), 3)
    prediction_widget.value = bgr8_to_jpeg(prediction)

# Create JetRacerROSOnnxRunner with dynamic slider evaluation
runner = JetRacerROSOnnxRunner(
    session=session,
    input_name=input_name,
    output_name=output_name,
    car=car,
    stanley=stanley,
    k=lambda: k_stanley_slider.value,
    throttle=lambda: base_throttle_slider.value,
    brake_gain=lambda: brake_gain_slider.value,
    bias=lambda: steering_bias_slider.value,
    alpha=lambda: alpha_slider.value,
    on_frame=on_ros_frame
)

runner.running = False  # Paused until mode changed to Live Drive

# Subscribe to ROS Camera Topic (/csi_cam_0/image_raw) with queue_size=1 and non-blocking frame drop
topic_name = "/csi_cam_0/image_raw"
ros_sub = rospy.Subscriber(topic_name, Image, runner.image_callback, queue_size=1, buff_size=2**24)
print(f"[*] Subscribed to ROS Image Topic: {topic_name}")

def on_state_change(change):
    if change['new'] in ['Stanley Live Drive', 'PID Live Drive']:
        stanley.reset()
        pid.reset()
        runner.running = True
    else:
        runner.running = False
        car.throttle = 0.0
        car.steering = 0.0

state_widget.observe(on_state_change, names='value')

def on_reset_clicked(b):
    state_widget.value = 'Off'
    runner.running = False
    car.throttle = 0.0
    car.steering = 0.0
    stanley.reset()
    pid.reset()

reset_button.on_click(on_reset_clicked)

# Layout Setup
top_box = ipywidgets.VBox([
    prediction_widget,
    ipywidgets.HBox([state_widget, reset_button, load_config_button])
], layout=Layout(align_items='center', margin='0px 0px 15px 0px'))

stanley_box = ipywidgets.VBox([
    ipywidgets.HTML(value="<h4>Stanley & Dynamic Controls</h4>"),
    k_stanley_slider,
    base_throttle_slider,
    brake_gain_slider,
    steering_bias_slider,
    alpha_slider
], layout=Layout(margin='0px 20px 0px 0px'))

pid_box = ipywidgets.VBox([
    ipywidgets.HTML(value="<h4>PID Gains & Real-time Displays</h4>"),
    kp_slider,
    ki_slider,
    kd_slider,
    ipywidgets.HTML(value="<b>Live Outputs:</b>"),
    raw_x_disp,
    steering_disp,
    throttle_disp
])

controls_grid = ipywidgets.HBox([stanley_box, pid_box], layout=Layout(justify_content='space-around'))

display(ipywidgets.VBox([top_box, controls_grid]))
